## Install Dependencies

In [ ]:
!pip install langchain langchain-community langchain-text-splitters langchain-huggingface langchain-chroma pymupdf chromadb sentence-transformers langchain-groq

## Imports

In [ ]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

## Keys and Setup

In [ ]:
os.environ['GROQ_API_KEY'] = 'gsk_YOUR_API_KEY_HERE'
llm = ChatGroq(model='llama3-8b-8192')
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

## Load and Split PDF

In [ ]:
loader = PyMuPDFLoader("sample_knowledge.txt")
pages = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = splitter.split_documents(pages)

## Create ChromaDB Vector Store

In [ ]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

## Build RAG Chain

In [ ]:
template = """
Answer the question based ONLY on the following context:
{context}

Question: {question}
"""
prompt = PromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## Query the Chain

In [ ]:
answer = rag_chain.invoke("What is this document about?")
print(answer)